<a href="https://colab.research.google.com/github/apar-tech/rag-chatbot/blob/main/phase2/langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

try:

    !pip install -q \
    langgraph \
    chromadb \
    google-genai \
    groq

    print("Libraries installed successfully!")

except Exception as e:

    print("Installation Error:")
    print(e)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [ ]:
try:

    from typing import TypedDict

    from langgraph.graph import (
        StateGraph,
        START,
        END
    )

    print("LangGraph imports successful!")

except Exception as e:

    print("Import Error:")
    print(e)

LangGraph imports successful!


In [ ]:
try:

    class GraphState(TypedDict):

        question: str

        query_embedding: list

        retrieved_chunks: list

        answer: str

    print("GraphState created successfully!")

except Exception as e:

    print("State Error:")
    print(e)

GraphState created successfully!


In [ ]:
try:

    from google import genai
    from google.colab import userdata

    gemini_client = genai.Client(
        api_key=userdata.get("geminikey")
    )

    print("Gemini connected successfully!")

except Exception as e:

    print("Gemini Connection Error:")
    print(e)


Gemini connected successfully!


In [ ]:
import pickle
import chromadb

with open("chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

with open("chunk_embeddings.pkl", "rb") as f:
    chunk_embeddings = pickle.load(f)

client = chromadb.PersistentClient(
    path="./digital_chromadb"
)

collection = client.get_or_create_collection(
    name="digital_docs"
)

collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunk_embeddings
)

print("Records:", collection.count())

Records: 111


In [ ]:
try:

    from groq import Groq
    from google.colab import userdata

    groq_client = Groq(
        api_key=userdata.get("groqkey")
    )

    print("Groq connected successfully!")

except Exception as e:

    print("Groq Connection Error:")
    print(e)

Groq connected successfully!


In [ ]:
try:

    def embed_question(state):

        question = state["question"]

        response = gemini_client.models.embed_content(
            model="models/gemini-embedding-001",
            contents=question
        )

        state["query_embedding"] = (
            response.embeddings[0].values
        )

        print("Question embedding generated!")

        return state

    print("Embed Question Node created!")

except Exception as e:

    print("Node Creation Error:")
    print(e)

Embed Question Node created!


In [ ]:
try:

    test_state = {
        "question": "What is machine learning?"
    }

    result = embed_question(test_state)

    print("Embedding Dimension:")
    print(len(result["query_embedding"]))

except Exception as e:

    print("Test Error:")
    print(e)

Question embedding generated!
Embedding Dimension:
3072


In [ ]:
try:

    def retrieve_chunks(state):

        results = collection.query(
            query_embeddings=[
                state["query_embedding"]
            ],
            n_results=5
        )

        state["retrieved_chunks"] = (
            results["documents"][0]
        )

        print("Top 5 chunks retrieved!")

        return state

    print("Retrieve Chunks Node created!")

except Exception as e:

    print("Node Creation Error:")
    print(e)

Retrieve Chunks Node created!


In [ ]:
try:

    state = {
        "question": "What is machine learning?"
    }

    state = embed_question(state)

    state = retrieve_chunks(state)

    print("\nRetrieved Chunks:")
    print(len(state["retrieved_chunks"]))

except Exception as e:

    print("Test Error:")
    print(e)

Question embedding generated!
Top 5 chunks retrieved!

Retrieved Chunks:
5


In [ ]:
try:

    for i, chunk in enumerate(
        state["retrieved_chunks"],
        start=1
    ):

        print("=" * 60)
        print(f"Chunk {i}")
        print(chunk[:500])
        print()

except Exception as e:

    print(e)

Chunk 1
Rule-based models
Rule-based machine learning (RBML) is a branch of machine learning that automatically discovers and learns 'rules' from data. It provides interpretable models, making it useful for decision-making in fields like healthcare, fraud detection, and cybersecurity. Key RBML techniques includes learning classifier systems, association rule learning, artificial immune systems, and other similar models. These methods extract patterns from data and evolve rules over time.

Training model

Chunk 2
Theory
A core objective of a learner is to generalise from its experience. Generalisation in this context is the ability of a learning machine to perform accurately on new, unseen examples/tasks after having experienced a learning data set. The training examples come from some generally unknown probability distribution (considered representative of the space of occurrences) and the learner has to build a general model about this space that enables it to produce sufficiently acc

In [ ]:
try:

    def generate_answer(state):

        context = "\n\n".join(
            state["retrieved_chunks"]
        )

        prompt = f"""
You are a networking assistant.

Answer only using the provided context.

If the answer is not available in the context,
respond with:

I could not find that information in the knowledge base.

Context:
{context}

Question:
{state["question"]}
"""

        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        state["answer"] = (
            response.choices[0].message.content
        )

        print("Answer generated successfully!")

        return state

    print("Generate Answer Node created!")

except Exception as e:

    print("Node Creation Error:")
    print(e)

Generate Answer Node created!


In [ ]:
try:

    state = {
        "question": "What is  machine learning?"
    }

    state = embed_question(state)

    state = retrieve_chunks(state)

    state = generate_answer(state)

    print("\nFinal Answer:\n")
    print(state["answer"])

except Exception as e:

    print("Pipeline Error:")
    print(e)

Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!

Final Answer:

Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed.


In [ ]:
try:

    graph_builder = StateGraph(GraphState)

    graph_builder.add_node(
        "embed_question",
        embed_question
    )

    graph_builder.add_node(
        "retrieve_chunks",
        retrieve_chunks
    )

    graph_builder.add_node(
        "generate_answer",
        generate_answer
    )

    print("Nodes added successfully!")

except Exception as e:

    print("Graph Creation Error:")
    print(e)

Nodes added successfully!


In [ ]:
try:

    graph_builder.add_edge(
        START,
        "embed_question"
    )

    graph_builder.add_edge(
        "embed_question",
        "retrieve_chunks"
    )

    graph_builder.add_edge(
        "retrieve_chunks",
        "generate_answer"
    )

    graph_builder.add_edge(
        "generate_answer",
        END
    )

    print("Edges connected successfully!")

except Exception as e:

    print("Edge Error:")
    print(e)


Edges connected successfully!


In [ ]:
try:

    graph = graph_builder.compile()

    print("LangGraph compiled successfully!")

except Exception as e:

    print("Compilation Error:")
    print(e)

LangGraph compiled successfully!


In [ ]:
try:

    result = graph.invoke(
        {   "question":
             "What is the purpose of AI?"
        }
    )

    print("\nFinal Answer:\n")
    print(result["answer"])

except Exception as e:

    print("Execution Error:")
    print(e)


Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!

Final Answer:

The purpose of AI is to develop and study methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals, performing tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making.


In [ ]:
result = graph.invoke(
    {
        "question":
        "What is the difference between machine learning and deep learning?"
    }
)

print(result["answer"])


Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!
Machine learning is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data. Deep learning, on the other hand, is a class of machine learning algorithms that uses a hierarchy of layers to transform input data into a progressively more abstract and composite representation, typically using multi-layered neural networks. In other words, all deep learning is machine learning, but not all machine learning is deep learning. Deep learning is a specific type of machine learning that focuses on utilizing multilayered neural networks to perform tasks such as classification, regression, and representation learning.


In [ ]:
result = graph.invoke(
    {
        "question":
        "What is the difference between machine language and human language?"
    }
)

print(result["answer"])

Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!
I could not find that information in the knowledge base.
